In [1]:
import os
import hashlib
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_date, datediff, when, udf, round,
    upper, trim, regexp_replace
)
from pyspark.sql.types import StringType

builder = SparkSession.builder \
    .appName("hospital-silver") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.shuffle.partitions", "8")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

PROJECT_BASE = os.path.expanduser(
    "~/data-engineering-pathway/projects/hospital-lakehouse"
)
DELTA_BASE = f"{PROJECT_BASE}/delta"

# Create Silver directories
os.makedirs(f"{DELTA_BASE}/silver/patients",   exist_ok=True)
os.makedirs(f"{DELTA_BASE}/silver/admissions", exist_ok=True)

print(f"Spark OK  : {spark.version}")
print(f"Delta base: {DELTA_BASE}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/26 01:28:41 WARN Utils: Your hostname, M-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.100.4 instead (on interface en0)
26/04/26 01:28:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/malone/data-engineering-pathway/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/malone/.ivy2.5.2/cache
The jars for the packages stored in: /Users/malone/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-57f91070-9e1e-41e0-8bde-164f7d12a13c;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.2.0 in central
	found io.delta#delta-storage;4.2.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.1 in central
	found org.slf4j#slf4j-api;2.0

Spark OK  : 4.1.1
Delta base: /Users/malone/data-engineering-pathway/projects/hospital-lakehouse/delta


In [2]:
# Read from Bronze Delta tables
bronze_patients   = spark.read.format("delta").load(f"{DELTA_BASE}/bronze/patients")
bronze_admissions = spark.read.format("delta").load(f"{DELTA_BASE}/bronze/admissions")

print(f"bronze_patients   loaded : {bronze_patients.count()} rows")
print(f"bronze_admissions loaded : {bronze_admissions.count()} rows")
bronze_patients.show(3)
bronze_admissions.show(3, truncate=False)

bronze_patients   loaded : 55500 rows
bronze_admissions loaded : 55500 rows


+------------+-------------+---+------+----------+
|admission_id| patient_name|age|gender|blood_type|
+------------+-------------+---+------+----------+
|           0|Bobby JacksOn| 30|  Male|        B-|
|           1| LesLie TErRy| 62|  Male|        A+|
|           2|  DaNnY sMitH| 76|Female|        A-|
+------------+-------------+---+------+----------+
only showing top 3 rows
+------------+-----------------+-----------------+--------------+-----------------------+------------------+------------------+-----------+--------------+--------------+----------+------------+
|admission_id|medical_condition|date_of_admission|doctor        |hospital               |insurance_provider|billing_amount    |room_number|admission_type|discharge_date|medication|test_results|
+------------+-----------------+-----------------+--------------+-----------------------+------------------+------------------+-----------+--------------+--------------+----------+------------+
|17179869184 |Obesity          |2023-

In [3]:
# SHA256 hash function — one-way, irreversible
def hash_name(name):
    if name is None:
        return None
    return hashlib.sha256(name.strip().lower().encode()).hexdigest()[:16]

hash_udf = udf(hash_name, StringType())

# Apply PII masking
silver_patients = bronze_patients \
    .withColumn("patient_id", hash_udf(col("patient_name"))) \
    .drop("patient_name") \
    .withColumn("gender", upper(trim(col("gender")))) \
    .withColumn("blood_type", upper(trim(col("blood_type")))) \
    .withColumn("age_group",
        when(col("age") < 18,  "Paediatric (0-17)")
        .when(col("age") < 36,  "Young Adult (18-35)")
        .when(col("age") < 61,  "Adult (36-60)")
        .otherwise("Senior (61+)")
    )

# Reorder columns cleanly
silver_patients = silver_patients.select(
    "admission_id",
    "patient_id",
    "age",
    "age_group",
    "gender",
    "blood_type"
)

print("Silver patients — PII masking applied:")
print(f"  patient_name column : REMOVED")
print(f"  patient_id column   : SHA256 hash (first 16 chars)")
silver_patients.show(5)
print(f"\nAge group distribution:")
silver_patients.groupBy("age_group") \
    .count().orderBy("count", ascending=False).show()

Silver patients — PII masking applied:
  patient_name column : REMOVED
  patient_id column   : SHA256 hash (first 16 chars)


+------------+----------------+---+-------------------+------+----------+
|admission_id|      patient_id|age|          age_group|gender|blood_type|
+------------+----------------+---+-------------------+------+----------+
|           0|4ccafce9615dbc9f| 30|Young Adult (18-35)|  MALE|        B-|
|           1|2561a5700f7c2a2a| 62|       Senior (61+)|  MALE|        A+|
|           2|183fb33f66843511| 76|       Senior (61+)|FEMALE|        A-|
|           3|90625d8f5620f665| 28|Young Adult (18-35)|FEMALE|        O+|
|           4|f7ff676ce52a62db| 43|      Adult (36-60)|FEMALE|       AB+|
+------------+----------------+---+-------------------+------+----------+
only showing top 5 rows

Age group distribution:
+-------------------+-----+
|          age_group|count|
+-------------------+-----+
|      Adult (36-60)|20598|
|       Senior (61+)|20370|
|Young Adult (18-35)|14416|
|  Paediatric (0-17)|  116|
+-------------------+-----+



In [4]:
# Cast date columns
silver_admissions = bronze_admissions \
    .withColumn("date_of_admission",
                to_date(col("date_of_admission"), "yyyy-MM-dd")) \
    .withColumn("discharge_date",
                to_date(col("discharge_date"), "yyyy-MM-dd"))

# Calculate length of stay in days
silver_admissions = silver_admissions \
    .withColumn("length_of_stay_days",
                datediff(col("discharge_date"), col("date_of_admission")))

# Standardise text columns
silver_admissions = silver_admissions \
    .withColumn("admission_type",
                upper(trim(col("admission_type")))) \
    .withColumn("medical_condition",
                upper(trim(col("medical_condition")))) \
    .withColumn("test_results",
                upper(trim(col("test_results"))))

# Add billing category
silver_admissions = silver_admissions \
    .withColumn("billing_category",
        when(col("billing_amount") < 5000,  "low")
        .when(col("billing_amount") < 20000, "mid")
        .otherwise("high")) \
    .withColumn("billing_amount",
                round(col("billing_amount"), 2))

# Data quality flag
silver_admissions = silver_admissions \
    .withColumn("data_quality",
        when(
            col("length_of_stay_days").isNull() |
            (col("length_of_stay_days") < 0) |
            col("billing_amount").isNull(),
            "investigate"
        ).otherwise("ok"))

print("Silver admissions transformations applied:")
print(f"  date columns cast to DateType")
print(f"  length_of_stay_days calculated")
print(f"  billing_category added")
print(f"  data_quality flag added")
silver_admissions.show(5, truncate=False)

print("\nLength of stay stats:")
silver_admissions.select("length_of_stay_days") \
    .summary("min", "max", "mean").show()

print("Billing category distribution:")
silver_admissions.groupBy("billing_category") \
    .count().orderBy("count", ascending=False).show()

print("Data quality breakdown:")
silver_admissions.groupBy("data_quality") \
    .count().show()

Silver admissions transformations applied:
  date columns cast to DateType
  length_of_stay_days calculated
  billing_category added
  data_quality flag added
+------------+-----------------+-----------------+----------------+--------------------------+------------------+--------------+-----------+--------------+--------------+-----------+------------+-------------------+----------------+------------+
|admission_id|medical_condition|date_of_admission|doctor          |hospital                  |insurance_provider|billing_amount|room_number|admission_type|discharge_date|medication |test_results|length_of_stay_days|billing_category|data_quality|
+------------+-----------------+-----------------+----------------+--------------------------+------------------+--------------+-----------+--------------+--------------+-----------+------------+-------------------+----------------+------------+
|0           |CANCER           |2024-01-31       |Matthew Smith   |Sons and Miller           |Blue Cros

In [5]:
# Write silver_patients
try:
    silver_patients.write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"{DELTA_BASE}/silver/patients")
    print(f"silver_patients written : {silver_patients.count()} rows")
except Exception as e:
    print(f"ERROR: {e}")

# Write silver_admissions
try:
    silver_admissions.write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"{DELTA_BASE}/silver/admissions")
    print(f"silver_admissions written : {silver_admissions.count()} rows")
except Exception as e:
    print(f"ERROR: {e}")

silver_patients written : 55500 rows
silver_admissions written : 55500 rows


In [6]:
# Reload from Delta to confirm persistence
sp = spark.read.format("delta").load(f"{DELTA_BASE}/silver/patients")
sa = spark.read.format("delta").load(f"{DELTA_BASE}/silver/admissions")

print("=" * 60)
print("SILVER LAYER — HOSPITAL COMPLETE")
print("=" * 60)
print(f"  silver_patients   : {sp.count()} rows")
print(f"  silver_admissions : {sa.count()} rows")
print(f"\n  PII status        : patient_name REMOVED")
print(f"  Patient identity  : SHA256 patient_id only")
print(f"\n  New columns added:")
print(f"    patients   : patient_id, age_group")
print(f"    admissions : length_of_stay_days, billing_category, data_quality")
print(f"\n  Silver path: {DELTA_BASE}/silver/")
print("=" * 60)
print("Silver complete. Bronze tables unchanged.")

SILVER LAYER — HOSPITAL COMPLETE
  silver_patients   : 55500 rows
  silver_admissions : 55500 rows

  PII status        : patient_name REMOVED
  Patient identity  : SHA256 patient_id only

  New columns added:
    patients   : patient_id, age_group
    admissions : length_of_stay_days, billing_category, data_quality

  Silver path: /Users/malone/data-engineering-pathway/projects/hospital-lakehouse/delta/silver/
Silver complete. Bronze tables unchanged.
